# Imports and Setup

In [1]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

INFO:2026-01-07 14:48:22,694:jax._src.xla_bridge:752: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-01-07 14:48:22,694 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-01-07 14:48:22,889 - INFO - SkewerBundle class definition complete


# Lines

In [2]:
from skewerjax.constants.lines import LINES, get_line, LineBundle

### Basic Usage: Fetch a single line by ID

In [3]:
# Get the Lyman-alpha transition
lya = get_line("HI-LyA")

print(f"ID: {lya.id}")
print(f"Species: {lya.species}")
print(f"Name: {lya.name}")
print(f"Rest wavelength: {lya.lambda0_A:.4f} Å")
print(f"Oscillator strength: {lya.f_osc:.4f}")
print(f"Einstein coefficient: {lya.A21_Hz:.4e} Hz = {lya.A21_GHz:.4f} GHz")
print(f"Mass: {lya.mass_amu:.4f} amu")

ID: HI-LyA
Species: HI
Name: Lyman-α
Rest wavelength: 1215.6701 Å
Oscillator strength: 0.4164
Einstein coefficient: 6.2650e+08 Hz = 0.6265 GHz
Mass: 1.0079 amu


### Explore the Global LINES Database

In [4]:
print(f"Total lines loaded: {len(LINES)}")

for line_id, line in sorted(LINES.items(), key=lambda x: x[1].lambda0_A):
    print(f"  Name:{line_id:11s} λ={line.lambda0_A:8.4f} Å. Species: {line.species:7s} f={line.f_osc:.4f} ")

Total lines loaded: 81
  Name:HeII-256    λ=256.3170 Å. Species: HeII    f=0.0790 
  Name:HeII-304    λ=303.7822 Å. Species: HeII    f=0.4160 
  Name:HeI-516     λ=515.6165 Å. Species: HeI     f=0.0154 
  Name:HeI-522     λ=522.2128 Å. Species: HeI     f=0.0307 
  Name:HeI-537     λ=537.0296 Å. Species: HeI     f=0.0752 
  Name:HeI-584     λ=584.3340 Å. Species: HeI     f=0.2850 
  Name:HI-LyLim    λ=911.7633 Å. Species: HI      f=0.0000 
  Name:HI-Ly31     λ=912.6450 Å. Species: HI      f=0.0000 
  Name:HI-Ly30     λ=912.7030 Å. Species: HI      f=0.0001 
  Name:HI-Ly29     λ=912.7680 Å. Species: HI      f=0.0001 
  Name:HI-Ly28     λ=912.8390 Å. Species: HI      f=0.0001 
  Name:HI-Ly27     λ=912.9180 Å. Species: HI      f=0.0001 
  Name:HI-Ly26     λ=913.0060 Å. Species: HI      f=0.0001 
  Name:HI-Ly25     λ=913.1040 Å. Species: HI      f=0.0001 
  Name:HI-LyW      λ=913.2150 Å. Species: HI      f=0.0001 
  Name:HI-LyY      λ=913.3390 Å. Species: HI      f=0.0001 
  Name:HI-LyC    

### Load some selected absorption lines using the LineBundle class

In [5]:
line_bundle = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

print(f"Lines: {line_bundle.n_lines}, Species: {line_bundle.n_species}")
print(f"Species keys: {line_bundle.species_keys}")

for ii in range(line_bundle.n_lines):
    print(f"{ii:2d} {line_bundle.lines[ii].id} {line_bundle.lines[ii].species} {line_bundle.lines[ii].lambda0_A} {line_bundle.lines[ii].f_osc} {line_bundle.lines[ii].A21_GHz} {line_bundle.lines[ii].mass_amu}")

print(f"HI lines: {line_bundle.lines_for_species('HI')}")
print(f"SiIII lines: {line_bundle.lines_for_species('SiIII')}")

print(f"HI indices: {line_bundle.indices_for_species('HI')}")
print(f"SiIII indices: {line_bundle.indices_for_species('SiIII')}")

print(f"Lambda0: {line_bundle.lambda0_A}")
print(f"A21: {line_bundle.A21_GHz}")
print(f"Mass: {line_bundle.mass_amu}")

Lines: 3, Species: 2
Species keys: ['HI', 'SiIII']
 0 HI-LyA HI 1215.6701 0.4164 0.6265000000000001 1.00794
 1 HI-LyB HI 1025.7223 0.07912 0.1897 1.00794
 2 SiIII-1206 SiIII 1206.5 1.63 2.48 28.0855
HI lines: [Line(id='HI-LyA', species='HI', name='Lyman-α', lambda0_A=1215.6701, f_osc=0.4164, A21_Hz=626500000.0, mass_amu=1.00794), Line(id='HI-LyB', species='HI', name='Lyman-β', lambda0_A=1025.7223, f_osc=0.07912, A21_Hz=189700000.0, mass_amu=1.00794)]
SiIII lines: [Line(id='SiIII-1206', species='SiIII', name='Si III 1206', lambda0_A=1206.5, f_osc=1.63, A21_Hz=2480000000.0, mass_amu=28.0855)]
HI indices: [0, 1]
SiIII indices: [2]
Lambda0: [1215.67   1025.7223 1206.5   ]
A21: [0.6265 0.1897 2.48  ]
Mass: [ 1.00794  1.00794 28.0855 ]
